In [ ]:
import gzip
import re
import os

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [ ]:
RAW_DIR = "data/ContentWiseImpressions/ContentWiseImpressions/data/ContentWiseImpressions/CW10M-CSV"
OUT_DIR = "data/ContentWiseImpressions/pre-processed"

MIN_CHOICES_PER_ITEM = 4
MIN_CHOICES_PER_USER = 5
SLATE_SIZE = 32

In [ ]:
with gzip.open(f"{RAW_DIR}/interactions.csv.gz") as f:
    interactions = pd.read_csv(f)

with gzip.open(f"{RAW_DIR}/impressions-direct-link.csv.gz") as f:
    impressions_direct_link = pd.read_csv(f)

In [ ]:
df_pos_interactions = (
    impressions_direct_link
    .merge(
        interactions,
        on=["recommendation_id"],
        how="inner"
    )
)

df_pos_interactions = df_pos_interactions[
    df_pos_interactions["vision_factor"]>0
]

In [ ]:
item_choice_counts = df_pos_interactions["series_id"].value_counts()
rare_items = item_choice_counts[item_choice_counts < MIN_CHOICES_PER_ITEM].index

print(f"Dropping {len(rare_items)} of {len(item_choice_counts)} items that were chosen fewer than {MIN_CHOICES_PER_ITEM} times")

df_pos_interactions = df_pos_interactions[~df_pos_interactions["series_id"].isin(rare_items)]

In [ ]:
df_neg_interactions = interactions[~interactions["recommendation_id"].isin(df_pos_interactions["recommendation_id"].values)]

df_neg_interactions = df_neg_interactions[df_neg_interactions["recommendation_id"]!=-1].drop_duplicates("recommendation_id", keep="first")

df_neg_interactions = df_neg_interactions.merge(impressions_direct_link, on="recommendation_id")

df_neg_interactions["series_id"] = np.nan

In [ ]:
columns = [
    "utc_ts_milliseconds",
    "user_id",
    "series_id",
    "recommended_series_list",
]

df = pd.concat([df_pos_interactions[columns], df_neg_interactions[columns]], ignore_index=True)

df = df.sort_values(by=["user_id", "utc_ts_milliseconds"]).reset_index(drop=True)

In [ ]:
df["series_id"] = df["series_id"].fillna(-1).astype(int)

df["recommended_series_list"] = df["recommended_series_list"].apply(
    lambda x: [-1] + list(map(int, re.findall(r"\d+", x)))
)

In [ ]:
df_num_pos_obs_per_user = df[df["series_id"]!=-1].groupby("user_id").size().reset_index(name="num_pos_obs_this_user")

df = df.merge(df_num_pos_obs_per_user, on="user_id")

df = df[df["num_pos_obs_this_user"] >= MIN_CHOICES_PER_USER]

In [ ]:
never_chosen_items = set(np.concatenate(df["recommended_series_list"].values)) - set(df["series_id"].unique())

df["recommended_series_list"] = df["recommended_series_list"].apply(
    lambda x: [y for y in x if y not in never_chosen_items]
)

assert df.apply(
    lambda row: row["series_id"] in row["recommended_series_list"],
    axis=1
).all(), "Not all series_ids are in the recommended_series_list"

In [ ]:
df["recommended_series_list"] = df["recommended_series_list"].apply(
    lambda x: x[:SLATE_SIZE]
)

df["recommended_series_list"] = df["recommended_series_list"].apply(
    lambda x: x + [-2] * (SLATE_SIZE - len(x))
)

df["recommended_series_list"] = df.apply(
    lambda x: (
        x.recommended_series_list[:-1] + [x.series_id] if not x.series_id in x.recommended_series_list
        else x.recommended_series_list
    ),
    axis=1,
)

In [ ]:
users = df["user_id"].values
choices = df["series_id"].values
options = np.array(df["recommended_series_list"].tolist())

item_le = LabelEncoder()
item_le.fit(np.unique(np.concatenate([choices, options.flatten()])))
choices_transformed = item_le.transform(choices)
options_transformed = item_le.transform(options.flatten()).reshape(options.shape)

user_le = LabelEncoder()
user_le.fit(users)
users_transformed = user_le.transform(users)

print(f"Padding ID: {item_le.transform([-2])[0]}, no-choice ID: {item_le.transform([-1])[0]}")

In [ ]:
item_types = interactions[["series_id", "item_type"]].drop_duplicates()

assert set(item_le.classes_).difference([-1, -2]).issubset(set(item_types["series_id"].unique())), "Not all series in df have an item type in item_type"

item_types = (
    item_types.set_index("series_id")
    .loc[
        list(set(item_le.classes_).intersection(set(item_types["series_id"].unique())))
    ].reset_index()
)

dupl_idx = (
    item_types[item_types.duplicated(subset=["series_id"], keep=False)]
    .query("item_type == 1 or item_type == 2")
    .index
)

item_types = item_types.drop(dupl_idx, axis=0)

item_types = item_types.set_index("series_id").reindex(
    item_le.classes_,
    fill_value=-1
).reset_index()

item_types_one_hot = pd.get_dummies(item_types["item_type"], prefix="item_type")

item_types_one_hot = item_types_one_hot.drop(columns=["item_type_-1"])

item_types_one_hot.columns = [
    "movies",
    "movies_and_clips_in_series",
    "tv_movies_or_shows",
    "episodes_of_tv_series"
]

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

np.savez(
    f"{OUT_DIR}/pre-processed_CW10M.npz",
    users=users_transformed,
    options=options_transformed,
    choices=choices_transformed,
    item_types=item_types_one_hot.values,
)

print(f"Observations: {len(users_transformed)} ({(choices_transformed == 1).mean():.1%} without a choice)")
print(f"Users: {len(user_le.classes_)}")
print(f"Items: {len(item_le.classes_) - 2}")